# KEIBA-AI 券種選択モデル訓練

history.dbのデータからXGBoostで最適券種を学習します。

実行順: セル1→2→3→4→5→6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sqlite3, json, math, pickle, os
import numpy as np
import pandas as pd
from itertools import combinations
from collections import defaultdict

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
HIST_DB  = f'{BASE_DIR}/data/history.db'

conn = sqlite3.connect(HIST_DB)
rh = pd.read_sql('SELECT * FROM race_history', conn)
hh = pd.read_sql('SELECT * FROM horse_history ORDER BY race_id, place', conn)
conn.close()

print(f'race_history: {len(rh):,}件')
print(f'horse_history: {len(hh):,}件')
print(f'期間: {hh["date"].min()} ~ {hh["date"].max()}')
print(f'会場: {sorted(hh["racecourse"].unique())}')

In [ ]:
import math as _math

def safe_int(val):
    if val is None: return 0
    try:
        if _math.isnan(float(val)): return 0
    except (TypeError, ValueError):
        return 0
    return int(val)

def pop_to_odds(pop):
    """
    人気番号 → 単勝オッズの近似変換（改善版）
    JRAの実測値に基づく近似。旧 [2.5, 4.5, 7.0, ...] は粒度が粗すぎたため修正。
    1番人気: 2.8倍 / 2番人気: 5.5倍 / 3番人気: 9.0倍
    → gap12 = 5.5/2.8 = 1.96 となり「断然」条件の実態に近づく
    """
    base = [2.8, 5.5, 9.0, 13.0, 18.0, 25.0, 35.0, 50.0, 70.0, 90.0, 120.0]
    try:
        p = float(pop)
        if _math.isnan(p): return 25.0
        idx = min(int(p) - 1, len(base) - 1)
        return base[max(0, idx)]
    except (TypeError, ValueError):
        return 25.0

def calc_chaos_from_pop(horses_df):
    pops = horses_df['popularity'].values
    if len(pops) < 2: return 0.5
    import statistics
    try:
        valid = [float(p) for p in pops if p is not None and not _math.isnan(float(p))]
        if len(valid) < 2: return 0.5
        cv = statistics.stdev(valid) / (statistics.mean(valid) + 0.001)
        return min(1.0, cv / 2.0)
    except:
        return 0.5

records = []

for race_id, grp in hh.groupby('race_id'):
    grp = grp.sort_values('place')
    n = len(grp)
    if n < 5: continue
    top1 = grp[grp['place'] == 1]
    top3 = grp[grp['place'] <= 3]
    if len(top1) == 0 or len(top3) < 3: continue

    # popularity が NaN の行を除外してから人気順ソート
    grp_valid = grp.dropna(subset=['popularity'])
    by_pop = grp_valid.sort_values('popularity')
    fav1 = by_pop.iloc[0] if len(by_pop) > 0 else None
    fav2 = by_pop.iloc[1] if len(by_pop) > 1 else None
    fav3 = by_pop.iloc[2] if len(by_pop) > 2 else None
    if fav1 is None: continue

    tan_pay  = safe_int(top1['tansho_payout'].values[0])
    fuku_pay = 0
    if fav1['place'] <= 3:
        fp = grp[grp['horse_num'] == fav1['horse_num']]['fukusho_payout'].values
        fuku_pay = safe_int(fp[0]) if len(fp) > 0 else 0

    o1 = pop_to_odds(fav1['popularity'])
    o2 = pop_to_odds(fav2['popularity']) if fav2 is not None else 25.0
    o3 = pop_to_odds(fav3['popularity']) if fav3 is not None else 35.0

    winner_num = top1['horse_num'].values[0]
    top3_nums  = set(top3['horse_num'].values)
    second_num = grp[grp['place'] == 2]['horse_num'].values
    second_num = second_num[0] if len(second_num) > 0 else -1

    tan_roi  = (tan_pay / 100) if winner_num == fav1['horse_num'] else 0.0
    if tan_pay == 0: tan_roi = o1 if winner_num == fav1['horse_num'] else 0.0

    fav1_hit = fav1['place'] <= 3
    fuku_roi = (fuku_pay / 100) if (fav1_hit and fuku_pay > 0) else (1.1 if fav1_hit else 0.0)

    wide_hit = (fav1['horse_num'] in top3_nums and fav2 is not None and fav2['horse_num'] in top3_nums)
    wide_pay = max(150, int((o1 * o2) ** 0.5 * 45))
    wide_roi = wide_pay / 100 if wide_hit else 0.0

    ren_hit  = ({fav1['horse_num'], fav2['horse_num'] if fav2 is not None else -1} == {winner_num, second_num})
    ren_pay  = max(200, int((o1 * o2) ** 0.5 * 75))
    ren_roi  = ren_pay / 100 if ren_hit else 0.0

    fav_nums = {fav1['horse_num']}
    if fav2 is not None: fav_nums.add(fav2['horse_num'])
    if fav3 is not None: fav_nums.add(fav3['horse_num'])
    san_hit  = (len(top3_nums & fav_nums) >= 3)
    chaos = calc_chaos_from_pop(grp)
    # chaos_score に応じて三連複オッズ推定に傾斜をつける（修正版）
    if chaos < 0.30:    # 堅いレース
        san_pay = max(250, int((o1 * o2 * o3) ** 0.3 * 55))
    elif chaos < 0.55:  # 中荒れ
        san_pay = max(400, int((o1 * o2 * o3) ** 0.4 * 75))
    else:               # 混戦
        san_pay = max(700, int((o1 * o2 * o3) ** 0.5 * 110))
    san_roi  = san_pay / 100 if san_hit else 0.0

    gap12 = o2 / o1 if o1 > 0 else 2.0
    gap23 = o3 / o2 if o2 > 0 else 2.0

    rinfo    = rh[rh['race_id'] == race_id]
    distance = int(rinfo['distance'].values[0]) if len(rinfo) > 0 else 1600
    surface  = 1 if (len(rinfo) > 0 and rinfo['surface'].values[0] == '芝') else 0
    last3f   = float(rinfo['last_3f'].values[0] or 0) if len(rinfo) > 0 else 0.0

    records.append({'race_id':race_id,'n':n,'o1':o1,'o2':o2,'o3':o3,
        'gap12':gap12,'gap23':gap23,'chaos':chaos,'distance':distance,
        'surface':surface,'last3f':last3f,
        'tan_roi':tan_roi,'fuku_roi':fuku_roi,'wide_roi':wide_roi,
        'ren_roi':ren_roi,'san_roi':san_roi})

df = pd.DataFrame(records)
print('学習データ: ' + str(len(df)) + ' レース')
print('各券種のROI > 1.0 の割合:')
roi_cols = ['tan_roi','fuku_roi','wide_roi','ren_roi','san_roi']
for col in roi_cols:
    hit = (df[col] > 1.0).mean()
    avg = df[col].mean()
    print(f'  {col}: 的中率{hit:.1%}  平均ROI{avg:.3f}')

In [ ]:
print('=== 頭数別ROI ===')
df['n_bin'] = pd.cut(df['n'], bins=[0,8,11,14,20], labels=['少(~8)','中(9-11)','多(12-14)','大(15+)'])
print(df.groupby('n_bin', observed=False)[roi_cols].mean().round(3))

print('\n=== chaos別ROI ===')
df['chaos_bin'] = pd.cut(df['chaos'], bins=[0,0.3,0.5,0.7,1.0], labels=['低','中低','中高','高'])
print(df.groupby('chaos_bin', observed=False)[roi_cols].mean().round(3))

print('\n=== オッズ差(gap12)別ROI ===')
# gap12 = o2/o1。新 pop_to_odds では 1番人気=2.8倍→gap12の典型値は1.96
# 断然: gap12>2.0（1番人気が2番人気に比べ明確に抜けている）
# 接近: gap12<1.5（1〜2番人気が拮抗）
df['gap_bin'] = pd.cut(df['gap12'], bins=[0, 1.5, 2.0, 3.0, 20],
                        labels=['接近(~1.5)', '2強(1.5-2.0)', 'やや断然(2.0-3.0)', '断然(3.0+)'])
print(df.groupby('gap_bin', observed=False)[roi_cols].mean().round(3))
print('\n=== gap12 の分布 ===')
print(df['gap_bin'].value_counts().sort_index())

print('\n=== 芝/ダート別ROI ===')
print(df.groupby('surface')[roi_cols].mean().round(3))

df['best_bet'] = df[roi_cols].idxmax(axis=1).str.replace('_roi','')
print('\n=== ベスト券種の分布 ===')
print(df['best_bet'].value_counts())
print('総レース数: ' + str(len(df)))

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

FEATURES = ['n','o1','o2','o3','gap12','gap23','chaos','distance','surface','last3f']
X = df[FEATURES].fillna(0)

y_raw = df[roi_cols].idxmax(axis=1).str.replace('_roi','')
le = LabelEncoder()
y  = le.fit_transform(y_raw)
print('クラス: ' + str(le.classes_))
print('分布: ' + str(dict(zip(le.classes_, [(y==i).sum() for i in range(len(le.classes_))]))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, verbosity=0)

model.fit(X_train, y_train,
    eval_set=[(X_test, y_test)], verbose=False)

y_pred = model.predict(X_test)
print('\n=== 分類レポート ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))
# 合格基準: san の recall > 0.3
san_idx = list(le.classes_).index('san')
from sklearn.metrics import recall_score
san_recall = recall_score(y_test, y_pred, labels=[san_idx], average='macro')
print(f'\n三連複(san) recall: {san_recall:.3f}  (合格基準: > 0.30)')

fi = pd.DataFrame({'feature':FEATURES,'importance':model.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print('\n=== 特徴量重要度 ===')
print(fi.to_string(index=False))

In [ ]:
print('=== 実データに基づく条件別ROI ===')
# gap12>2.0 が「断然」の実態に合った定義（pop_to_odds修正後）
conditions = {
    '断然(gap12>2.0)':        df['gap12'] > 2.0,
    '2強(1.5<=gap12<=2.0)':   (df['gap12'] >= 1.5) & (df['gap12'] <= 2.0),
    '接近(gap12<1.5)':         df['gap12'] < 1.5,
    '混戦(chaos>0.55)':        df['chaos'] > 0.55,
    '少頭数(n<=8)':             df['n'] <= 8,
    '多頭数(n>=14)':            df['n'] >= 14,
    '混戦+多頭数':              (df['chaos'] > 0.55) & (df['n'] >= 12),
    '中オッズ(3<=o1<=12)':     (df['o1'] >= 3) & (df['o1'] <= 12),
}
for cond_name, mask in conditions.items():
    sub = df[mask]
    if len(sub) < 10: continue
    roi_mean = sub[roi_cols].mean()
    best = roi_mean.idxmax().replace('_roi','')
    print(cond_name + ' (n=' + str(len(sub)) + '件):')
    for col in roi_cols:
        print(f'  {col.replace("_roi","")}: {roi_mean[col]:.3f}', end='')
    print(' -> ベスト: ' + best)

In [ ]:
import pickle

with open(f'{BASE_DIR}/data/bet_selector_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open(f'{BASE_DIR}/data/bet_selector_le.pkl', 'wb') as f:
    pickle.dump(le, f)

meta = {
    'features': FEATURES,
    'classes': list(le.classes_),
    'trained_on': str(len(df)) + '件',
    'date_range': hh['date'].min() + ' ~ ' + hh['date'].max()
}
with open(f'{BASE_DIR}/data/bet_selector_meta.json', 'w') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('モデル保存完了: bet_selector_model.pkl')
print('使い方:')
print('  features = [n, o1, o2, o3, gap12, gap23, chaos, distance, surface, last3f]')
print('  best_type = le.inverse_transform(model.predict([features]))[0]')